# Study 09 · 07 · E6 · Case definition and outcome definition

- **E6a:** share of confirmed notifications by season (national file, codes C and P).
- **E6b:** the primary pair refitted on **confirmed + probable** cases (2000–2024 from the national file; 2025–2026 as in the confirmed series). Evaluated on 2009–2025 (protocol D5), plus 2009–2026 for reference.
- **E6c:** the outbreak threshold **without epidemic years** in the reference (protocol D4, decided from past years only), applied to the confirmed-case forecasts.
- **E6d:** a fixed incidence rule (≥ 10 confirmed cases per 100,000 per week, as in the paper), using the panel's population for 2017–2025, so seasons 2019–2025. The INEI 2018–2026 file the paper used is not in `data 5`.

E6c and E6d reuse the stored count forecasts (mean and dispersion) and refit only the chronological calibrator, as the paper's label sensitivity does.

In [1]:
import os
import sys
from pathlib import Path

os.environ.update({"DENGUEKAY_XGB_DEVICE": "cpu", "DENGUEKAY_XGB_NJOBS": "2", "OMP_NUM_THREADS": "2",
                   "OPENBLAS_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "MLFLOW_DISABLE_AGENT_HINT": "1"})
STUDY = Path.cwd().parent
sys.path.insert(0, str(STUDY / "code"))

import numpy as np
import pandas as pd

import common as c

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)
import matplotlib.pyplot as plt

import experiments as ex
import run_extra as rx

FIG = STUDY / "figures"
RES = c.OUTPUT / "resultados"
FIG.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)
base = pd.read_csv(c.OUTPUT / "baselines.csv.gz", dtype={"ubigeo": str}, parse_dates=["semana_inicio", "origen_cierre"])
from denguekay.data.seasons import assign_season
inputs, _ = c.load_inputs(validate=False)
arms = {name: c.read_arm(name, columns=ex.CALIBRATION_COLUMNS) for name in ("b3_obs", "historia_geo")}
support = arms["b3_obs"][arms["b3_obs"].semilla.eq(17)].drop(columns=["semilla", "mu_modelo", "alpha_usada", "probabilidad_cruda", "probabilidad_calibrada"]).reset_index(drop=True)

## E6a · Confirmed share by season (Piura, national file)

In [2]:
raw = pd.read_csv(rx.NATIONAL, usecols=["departamento", "tipo_dx", "ano", "semana"], dtype=str)
raw = raw[raw.departamento.str.strip().str.upper().eq("PIURA")]
raw["temporada"] = assign_season(raw.ano.astype(int), raw.semana.astype(int), 35)
share = raw.groupby(["temporada", raw.tipo_dx.str.strip().str.upper()]).size().unstack(fill_value=0)
share["confirmed_%"] = (100 * share["C"] / share.sum(axis=1)).round(1)
share.to_csv(RES / "e6a_confirmados_por_temporada.csv")
share.T

temporada,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
tipo_dx,,,,,,,,,,,,,,,,,,,,,,,,,,,
C,420.0,371.0,16.0,110.0,11.0,10.0,588.0,279.0,1432.0,4241.0,8451.0,141.0,1098.0,2095.0,2190.0,17058.0,6586.0,42259.0,872.0,87.0,106.0,2962.0,12549.0,74657.0,36158.0,2190.0,111.0
P,826.0,12675.0,109.0,1610.0,38.0,38.0,239.0,31.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,2499.0,1453.0,1990.0,222.0,2.0,15.0,2.0,163.0,113.0,2.0,0.0,0.0
confirmed_%,33.7,2.8,12.8,6.4,22.4,20.8,71.1,90.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,87.2,81.9,95.5,79.7,97.8,87.6,99.9,98.7,99.8,100.0,100.0,100.0


## E6b · Confirmed + probable cases

In [3]:
cp = {name: c.read_arm(name, columns=ex.PRED_COLUMNS, folder=rx.CP_DIR) for name in ("b3_obs", "historia_geo")}
cp_labels = pd.read_csv(rx.CP_DIR / "etiquetas.csv.gz", dtype={"ubigeo": str}, parse_dates=["semana_inicio"])
cp_support = cp["b3_obs"][cp["b3_obs"].semilla.eq(17)][c.archive.OBSERVATIONS].reset_index(drop=True)
cp_base = c.baselines(cp_support, cp_labels)
rows = []
for label, seasons in (("2009-2025 (protocol)", tuple(range(2009, 2026))), ("2009-2026", c.SEASONS)):
    a, b = ex.sse(cp["b3_obs"], seasons=seasons), ex.sse(cp["historia_geo"], seasons=seasons)
    pp = ex.sse(cp_base, "persistencia_prob", seasons=seasons)
    rows += [ex.contrast(a, b).assign(seasons=label, comparison="b3 vs history+geography"),
             ex.contrast(a, pp).assign(seasons=label, comparison="b3 vs probabilistic persistence")]
e6b = pd.concat(rows)[["seasons", "comparison", "horizonte", "brier_a", "brier_b", "cambio_agrupado_%", "ic95_agrupado", "cambio_igual_peso_%", "temporadas_a_mejor"]]
e6b.to_csv(RES / "e6b_confirmados_probables.csv", index=False)
prev = ex.seasons_only(cp_support)
print("C+P elevated-week prevalence (h4):", round(100 * prev[prev.horizonte.eq(4)].brote.mean(), 2), "%")
e6b.round(3)

C+P elevated-week prevalence (h4): 7.76 %


,seasons,comparison,horizonte,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,temporadas_a_mejor
0,2009-2025 (protocol),b3 vs history+geography,2,0.030,0.030,-0.724,"(-1.99, 0.5)",-1.088,13
1,2009-2025 (protocol),b3 vs history+geography,3,0.034,0.034,-0.826,"(-2.46, 0.62)",-1.178,12
2,2009-2025 (protocol),b3 vs history+geography,4,0.037,0.038,-0.832,"(-3.07, 1.04)",-0.884,11
0,2009-2025 (protocol),b3 vs probabilistic persistence,2,0.030,0.031,-2.524,"(-6.84, 3.4)",8.127,8
1,2009-2025 (protocol),b3 vs probabilistic persistence,3,0.034,0.035,-4.793,"(-9.34, 1.91)",7.518,9
2,2009-2025 (protocol),b3 vs probabilistic persistence,4,0.037,0.040,-6.360,"(-11.19, 1.46)",9.095,10
0,2009-2026,b3 vs history+geography,2,0.029,0.029,-0.767,"(-1.99, 0.44)",-1.139,14
1,2009-2026,b3 vs history+geography,3,0.033,0.033,-0.856,"(-2.46, 0.59)",-1.228,13
2,2009-2026,b3 vs history+geography,4,0.036,0.036,-0.845,"(-3.0, 1.0)",-0.927,12
0,2009-2026,b3 vs probabilistic persistence,2,0.029,0.029,-2.379,"(-6.68, 3.53)",8.000,8


## E6c · Threshold without epidemic years

In [4]:
tau, years_used = ex.thresholds_without_epidemic_years(inputs["casos"], support)
k_new = np.maximum(np.floor(tau) + 1, 2)
relabelled = {}
for name, frame in arms.items():
    parts = []
    for seed, f in frame.groupby("semilla"):
        f = f.reset_index(drop=True)
        t, _ = ex.thresholds_without_epidemic_years(inputs["casos"], f)
        parts.append(ex.relabel(f, np.maximum(np.floor(t) + 1, 2)))
    relabelled[name] = pd.concat(parts, ignore_index=True)
pp_new = ex.relabel_persistence(support, k_new)
a, b = ex.sse(relabelled["b3_obs"], "p_nueva"), ex.sse(relabelled["historia_geo"], "p_nueva")
pp = ex.sse(pp_new, "p_nueva")
e6c = pd.concat([ex.contrast(a, b).assign(comparison="b3 vs history+geography"),
                 ex.contrast(a, pp).assign(comparison="b3 vs probabilistic persistence")])
e6c.to_csv(RES / "e6c_umbral_sin_epidemias.csv", index=False)
old = ex.seasons_only(support)
new = ex.seasons_only(pp_new)
print("prevalence h4, standard vs without epidemic years:", round(100 * old[old.horizonte.eq(4)].brote.mean(), 2), "% vs",
      round(100 * new[new.horizonte.eq(4)].brote.mean(), 2), "%")
display(years_used[years_used.anio_objetivo.between(2009, 2026)])
e6c.round(3)

prevalence h4, standard vs without epidemic years: 7.57 % vs 10.78 %


,anio_objetivo,anios_epidemicos_excluidos,anios_referencia
2,2009,"[2000, 2006, 2008]","[2002, 2003, 2004, 2005, 2007]"
3,2010,"[2006, 2008, 2009]","[2002, 2003, 2004, 2005, 2007]"
4,2011,"[2008, 2009, 2010]","[2003, 2004, 2005, 2006, 2007]"
5,2012,"[2008, 2009, 2010]","[2004, 2005, 2006, 2007, 2011]"
6,2013,"[2008, 2009, 2010, 2012]","[2004, 2005, 2006, 2007, 2011]"
7,2014,"[2008, 2009, 2010, 2013]","[2005, 2006, 2007, 2011, 2012]"
8,2015,"[2009, 2010, 2013, 2014]","[2006, 2007, 2008, 2011, 2012]"
9,2016,"[2009, 2010, 2014, 2015]","[2007, 2008, 2011, 2012, 2013]"
10,2017,"[2009, 2010, 2014, 2015, 2016]","[2007, 2008, 2011, 2012, 2013]"
11,2018,"[2009, 2010, 2015, 2016, 2017]","[2008, 2011, 2012, 2013, 2014]"


,horizonte,temporadas,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,ic95_igual_peso,temporadas_a_mejor,comparison
0,2,18,0.036,0.036,-0.628,"(-1.87, 0.35)",-1.034,"(-2.86, 0.51)",9,b3 vs history+geography
1,3,18,0.040,0.041,-0.588,"(-2.6, 0.85)",-0.985,"(-3.96, 1.85)",9,b3 vs history+geography
2,4,18,0.045,0.045,-0.376,"(-3.26, 1.81)",-0.112,"(-4.48, 4.91)",11,b3 vs history+geography
0,2,18,0.036,0.036,0.547,"(-3.79, 6.49)",8.390,"(0.64, 17.39)",7,b3 vs probabilistic persistence
1,3,18,0.040,0.040,-0.444,"(-5.33, 6.86)",10.701,"(0.09, 22.93)",8,b3 vs probabilistic persistence
2,4,18,0.045,0.045,-1.398,"(-7.27, 7.59)",13.777,"(-0.03, 29.93)",10,b3 vs probabilistic persistence


## E6d · Incidence rule (≥ 10 per 100,000 per week; seasons 2019–2025)

In [5]:
population = (pd.read_csv(c.archive.PANEL, dtype={"ubigeo": str}, usecols=["ubigeo", "anio", "poblacion"])
              .drop_duplicates(["ubigeo", "anio"]))
def incidence_k(frame, rate=10):
    pop = frame[["ubigeo", "anio"]].merge(population, on=["ubigeo", "anio"], how="left").poblacion.to_numpy(float)
    return np.ceil(rate * pop / 1e5)
seasons_inc = tuple(range(2019, 2026))
keep = lambda f: f[c.season_of(f.bloque).between(2018, 2025)].reset_index(drop=True)
inc = {}
for name, frame in arms.items():
    parts = [ex.relabel(keep(f), incidence_k(keep(f))) for _, f in frame.groupby("semilla")]
    inc[name] = pd.concat(parts, ignore_index=True)
s = keep(support)
pp_inc = ex.relabel_persistence(s, incidence_k(s))
a, b = ex.sse(inc["b3_obs"], "p_nueva", seasons=seasons_inc), ex.sse(inc["historia_geo"], "p_nueva", seasons=seasons_inc)
pp = ex.sse(pp_inc, "p_nueva", seasons=seasons_inc)
e6d = pd.concat([ex.contrast(a, b).assign(comparison="b3 vs history+geography"),
                 ex.contrast(a, pp).assign(comparison="b3 vs probabilistic persistence")])
e6d.to_csv(RES / "e6d_incidencia.csv", index=False)
e6d.round(3)

,horizonte,temporadas,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,ic95_igual_peso,temporadas_a_mejor,comparison
0,2,7,0.045,0.044,0.839,"(-0.9, 2.41)",2.387,"(0.19, 5.26)",1,b3 vs history+geography
1,3,7,0.052,0.051,1.578,"(-1.48, 4.53)",4.776,"(0.27, 11.37)",1,b3 vs history+geography
2,4,7,0.058,0.057,2.298,"(-1.1, 5.93)",7.099,"(0.44, 17.79)",1,b3 vs history+geography
0,2,7,0.045,0.045,-0.929,"(-9.58, 19.4)",4.336,"(-6.49, 19.65)",3,b3 vs probabilistic persistence
1,3,7,0.052,0.050,2.303,"(-11.44, 31.63)",10.235,"(-5.31, 30.97)",2,b3 vs probabilistic persistence
2,4,7,0.058,0.057,2.910,"(-13.75, 40.52)",15.146,"(-6.44, 45.74)",3,b3 vs probabilistic persistence
